## Overview
### Context
---
"You are a Senior Business Intelligence Analyst supporting the Payment Services team of an e-commerce platform.
The Business team wants to grow usage of strategic payment methods such as Wallet and BNPL, while the e-commerce platform also needs to maintain healthy order growth, user retention, seller participation, and profitability."

### Task
---
"You are provided with six months of order-level data. 

Your task is to analyze:

    - buyer
    - seller
    - category
    - payment behavior 

to help management decide:
    
<b style="color: red;">Which payment services should the company prioritize, for which user segments, and under what subsidy or growth strategy?</b>

>[!Notes]
>
> "Consider various factors and think critically about how the data can guide decision
>
> Feel free to propose any strategy derived from your analysis.
>
> Present your results and recommendations within 10 slides."

## Data

[data-case-study](raw_data/case_study_raw_data.csv)


### Data-Dictionary
| Field Name | Description | Sample Data |
| --- | --- | --- |
| **create_date** | The date when the order was placed. | 2024-05-20 |
| **order_id** | Unique identifier for each order. | CBZZWZ7C7CW7X5XW8X |
| **user_id** | Unique identifier for the user who placed the order. | A064540 |
| **state** | The user's city or address. | Hà Nội |
| **gender** | The gender of the user placing the order. | female |
| **category** | The product category of the item(s) purchased (e.g., electronics). | Beauty |
| **gross_merchant_value** | The total value of the order before any discounts. | 9 |
| **net_merchant_value** | The total value of the order after applying platform discounts. | 8.18 |
| **platform_discount** | The discount offered by the e-commerce platform (if any) on the order. | 0.82 |
| **seller_discount** | The discount offered by the seller (if any) on the order. | 0.45 |
| **user_payment_fee** | The fee charged to the buyer for the payment method used. | 0.13 |
| **payment_method** | The method used by the buyer to pay (e.g., BNPL, credit card, wallet). | BNPL |
| **total_user_payment** | The final amount paid by the buyer after all discounts and fees are applied. | 7.78 |


## DataFrame Config

In [5]:
import pandas as pd

data_path = "raw_data/case_study_raw_data.csv"
df_case_study = pd.read_csv(data_path)

## Start Analysing

### Up-coming Question

1. What are segmenation of customer are in the dataset ? 
    according to:
        - gmv
        - nmv
        - discount (platform_discount + seller_discount)
        - payment-method 
        - last_order

In [20]:
import pandas as pd
import numpy as np

df_case_study['create_date'] = pd.to_datetime(df_case_study['create_date'])

df_case_study['is_bnpl'] = df_case_study['payment_method'] == 'BNPL'
df_case_study['is_wallet'] = df_case_study['payment_method'] == 'Wallet'
df_case_study['total_discount'] = df_case_study['platform_discount'] + df_case_study['seller_discount']

df_user_stats = df_case_study.groupby('user_id').agg(
    order=('order_id', 'nunique'),
    nmv=('net_merchant_value', 'sum'),
    total_discount_sum=('total_discount', 'sum'),
    gmv_sum=('gross_merchant_value', 'sum'),
    bnpl_order=('is_bnpl', 'sum'),
    wallet_order=('is_wallet', 'sum'),
    last_order=('create_date', 'max')
).reset_index()

### Assumptions for User Segmentation

* **`new_user`**: The user has completed exactly 1 order.
* **`repeat_buyer`**: The user has completed more than 1 order.
* **`high_value_user`**: The user falls into the top 10% of all customers based on their total Net Merchant Value (NMV).
* **`BNPL_user`**: Buy Now, Pay Later (BNPL) was used for at least 50% of the user's total orders.
* **`wallet_user`**: A digital wallet was used for at least 50% of the user's total orders.
* **`discount_sensitive_user`**: The user's total combined discounts (platform + seller) amount to 10% or more of their total Gross Merchant Value (GMV).
* **`dormant_user`**: The user has not placed an order in 30 days or more, calculated relative to the most recent transaction date in the entire dataset.

In [24]:
df_segmented = df_user_stats.copy()
df_segmented['avg_basket'] = df_user_stats['nmv'] / df_user_stats['order']
df_segmented['discount_pct'] = (100.0 * df_user_stats['total_discount_sum']) / df_user_stats['gmv_sum']

nmv_rank = df_segmented['nmv'].rank(method='first', ascending=False)
df_segmented['value_decile'] = pd.qcut(nmv_rank, q=10, labels=False) + 1 

as_of = df_segmented['last_order'].max()
df_segmented['days_since_last'] = (as_of - df_segmented['last_order']).dt.days

df_segmented['nmv'] = df_segmented['nmv'].round(2)
df_segmented['avg_basket'] = df_segmented['avg_basket'].round(2)
df_segmented['discount_pct'] = df_segmented['discount_pct'].round(2)

# Apply segments
df_segmented['seg_frequency'] = np.where(df_segmented['order'] == 1, 'new_user', 'repeat_buyer')
df_segmented['seg_value'] = np.where(df_segmented['value_decile'] == 1, 'high_value_user', None)
df_segmented['seg_bnpl'] = np.where(df_segmented['bnpl_order'] >= df_segmented['order'] * 0.5, 'BNPL_user', None)
df_segmented['seg_wallet'] = np.where(df_segmented['wallet_order'] >= df_segmented['order'] * 0.5, 'wallet_user', None)
df_segmented['seg_discount'] = np.where(df_segmented['discount_pct'] >= 10, 'discount_sensitive_user', None)
df_segmented['seg_dormant'] = np.where(df_segmented['days_since_last'] >= 30, 'dormant_user', None)

final_cols = [
    'user_id', 
    'order', 
    'nmv', 
    'avg_basket', 
    'discount_pct', 
    'days_since_last',
    'seg_frequency', 
    'seg_value', 
    'seg_bnpl', 
    'seg_wallet', 
    'seg_discount', 
    'seg_dormant'
]

df_segmented = df_segmented[final_cols].sort_values('nmv', ascending=False)
df_segmented.to_csv("analyse_data_support/user_segmentation.csv", index=False)

In [14]:
df_segmented

,user_id,df_case_study,nmv,avg_basket,discount_pct,days_since_last,seg_frequency,seg_value,seg_bnpl,seg_wallet,seg_discount,seg_dormant
99,A0M3423,51,3545.40,69.52,0.44,6,repeat_buyer,high_value_user,NaN,NaN,NaN,NaN
467,A33NMN4,70,1969.58,28.14,0.91,0,repeat_buyer,high_value_user,NaN,NaN,NaN,NaN
333,A25MN6M,47,1099.07,23.38,2.29,6,repeat_buyer,high_value_user,NaN,NaN,NaN,NaN
454,A3321M2,237,981.34,4.14,13.48,1,repeat_buyer,high_value_user,NaN,NaN,discount_sensitive_user,NaN
120,A104N31,30,931.60,31.05,1.33,4,repeat_buyer,high_value_user,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
413,A3066L2,1,0.29,0.29,98.21,7,new_user,NaN,NaN,NaN,discount_sensitive_user,NaN
400,A30044N,1,0.29,0.29,103.78,8,new_user,NaN,NaN,NaN,discount_sensitive_user,NaN
46,A04N641,1,0.29,0.29,98.34,167,new_user,NaN,NaN,NaN,discount_sensitive_user,dormant_user
33,A03NM,1,0.29,0.29,97.08,17,new_user,NaN,BNPL_user,NaN,discount_sensitive_user,NaN
